### Parte 0: Configuración inicial y carga del corpus

Este bloque de código se encarga de definir la ruta donde se encuentran los archivos del corpus, cargar cada documento (`.txt`) en una lista para su posterior procesamiento, y verificar la existencia del directorio manejando posibles errores de lectura.

In [ ]:
import os

ruta_corpus = 'corpus_limpio'

docs = []
nombres = []

if not os.path.exists(ruta_corpus):
    print(f"Directorio no encontrado: {ruta_corpus}")
else:
    for nombre_archivo in sorted(os.listdir(ruta_corpus)):
        if nombre_archivo.endswith('.txt'):
            ruta_archivo = os.path.join(ruta_corpus, nombre_archivo)
            try:
                with open(ruta_archivo, 'r', encoding='utf-8') as f:
                    contenido = f.read()
                if contenido.strip():
                    docs.append(contenido)
                    nombres.append(nombre_archivo)
            except Exception as e:
                print(f"Error leyendo {nombre_archivo}: {e}")

    print(f"Documentos cargados: {len(docs)}")

### Parte 1: Cálculo de TF, DF, IDF y TF-IDF

Esta sección se enfoca en la extracción de características textuales clave utilizando las librerías `pandas` y `sklearn`. Los pasos son los siguientes:

1.  **Matriz de Términos (TF) y Frecuencia de Documentos (DF):**
    *   Se inicializa `CountVectorizer` para transformar los documentos en una matriz de frecuencias de términos (TF).
    *   Se obtiene la lista de todos los términos únicos (features).
    *   Se convierte la matriz TF a un DataFrame para una visualización más legible.
    *   Se calcula la Frecuencia de Documentos (DF), que indica cuántos documentos contienen cada término.
2.  **Cálculo de TF-IDF con `sklearn`:**
    *   Se inicializa `TfidfTransformer`, que por defecto aplica un escalado sublineal a TF y suaviza el cálculo de IDF.
    *   Se ajusta y transforma la matriz TF para obtener la matriz TF-IDF, que pondera la importancia de los términos considerando su frecuencia en el documento y su rareza en el corpus.
    *   Se convierte la matriz TF-IDF a un DataFrame para su inspección.
3.  **Visualización para Análisis:**
    *   Se combinan los valores de TF, DF, IDF y TF-IDF para un documento específico en un DataFrame consolidado.
    *   Este DataFrame se ordena por el valor TF-IDF para resaltar los términos más relevantes.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

vectorizador = CountVectorizer()
X = vectorizador.fit_transform(docs)
terminos = vectorizador.get_feature_names_out()

tf_df = pd.DataFrame(X.toarray(), columns=terminos)
tf_df.index = nombres

serie_df = (X > 0).sum(axis=0)
df_frecuencias = pd.DataFrame(serie_df.reshape(1, -1), columns=terminos, index=['DF'])

transformador_tfidf = TfidfTransformer()
matriz_tfidf = transformador_tfidf.fit_transform(X)

tfidf_df = pd.DataFrame(matriz_tfidf.toarray(), columns=terminos)
tfidf_df.index = nombres

valores_idf = transformador_tfidf.idf_

if len(nombres) > 0:
    primer_doc = nombres[0]
    datos_combinados = {
        'Término': terminos,
        'TF (Doc 1)': tf_df.loc[primer_doc].values,
        'DF': serie_df.A1,
        'IDF': valores_idf,
        'TF-IDF (Doc 1)': tfidf_df.loc[primer_doc].values
    }
    analisis_df = pd.DataFrame(datos_combinados)
    analisis_df = analisis_df.sort_values(by='TF-IDF (Doc 1)', ascending=False).reset_index(drop=True)
    print(f"Análisis TF, DF, IDF, TF-IDF para '{primer_doc}' (ordenado por TF-IDF):")
    analisis_df.head(20)

### Parte 2: Ranking de Documentos utilizando Similitud Coseno TF-IDF

Este bloque de código demuestra cómo utilizar los vectores TF-IDF calculados para clasificar documentos en función de su similitud con una consulta dada. Los pasos son:

1.  **Construcción del Vector de Consulta:**
    *   La consulta de ejemplo (`"lugares turisticos ecologico aventura"`) se transforma en un vector TF y luego en un vector TF-IDF, utilizando los mismos `vectorizador` y `transformador_tfidf` ajustados con el corpus.
2.  **Cálculo de Similitud Coseno:**
    *   Se utiliza `cosine_similarity` para medir la similitud entre el vector TF-IDF de la consulta y la matriz TF-IDF de los documentos. La similitud coseno produce un valor entre 0 y 1, donde 1 indica una similitud perfecta.
3.  **Generación del Ranking:**
    *   Se crea un DataFrame que combina los nombres de los documentos con sus respectivos scores de similitud.
    *   Los documentos se ordenan de mayor a menor score de similitud.
4.  **Visualización de Resultados:**
    *   Se muestra el ranking de documentos con el score de similitud coseno para cada uno.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

consulta = "lugares turisticos ecologico aventura"

consulta_tf = vectorizador.transform([consulta])
consulta_tfidf = transformador_tfidf.transform(consulta_tf)

puntajes_similitud = cosine_similarity(consulta_tfidf, matriz_tfidf)

ranking_tfidf = pd.DataFrame({
    'Documento': nombres,
    'Score Similitud': puntajes_similitud[0]
})
ranking_tfidf = ranking_tfidf.sort_values(by='Score Similitud', ascending=False).reset_index(drop=True)

print(f"Consulta: '{consulta}'\n")
print("Ranking por Similitud Coseno TF-IDF:")
ranking_tfidf.head(10)

### Parte 3: Implementación de BM25 y Ranking de Documentos

Esta sección implementa el algoritmo BM25 (Best Match 25) para calcular la relevancia de los documentos con respecto a una consulta. Incluye los siguientes pasos:

1.  **Tokenización:** Una función simple para convertir texto a minúsculas y dividirlo en palabras.
2.  **Longitudes de Documentos:** Calcula la longitud de cada documento tokenizado y la longitud promedio.
3.  **Cálculo de TF (Term Frequency):** Utiliza un `CountVectorizer` para obtener las frecuencias de los términos en cada documento.
4.  **Cálculo de DF (Document Frequency) e IDF de BM25:** Calcula la frecuencia de documentos para cada término y un IDF específico para BM25.
5.  **Parámetros de BM25:** Define los parámetros `k1` y `b` que controlan la saturación del TF y la normalización por longitud del documento.
6.  **Función de Score BM25:** Implementa la fórmula de BM25 para calcular el score de relevancia de un documento frente a una consulta.
7.  **Función de Ranking:** Procesa una consulta, calcula el score BM25 para todos los documentos y los ordena por relevancia.
8.  **Ejecución y Comparación:** Ejecuta el ranking con una consulta de ejemplo y muestra los resultados, incluyendo una comparación con el ranking de TF-IDF.

In [ ]:
from math import log
import re

def tokenizar(texto):
    return re.findall(r'\b\w+\b', texto.lower())

docs_tokenizados = [tokenizar(doc) for doc in docs]
longitudes = [len(d) for d in docs_tokenizados]
longitud_promedio = sum(longitudes) / len(longitudes) if longitudes else 1

matriz_tf_bm25 = vectorizador.fit_transform(docs)
terminos_bm25 = vectorizador.get_feature_names_out()

dicts_tf = []
for fila in matriz_tf_bm25.toarray():
    dicts_tf.append({t: f for t, f in zip(terminos_bm25, fila) if f > 0})

df_bm25_array = (matriz_tf_bm25 > 0).sum(axis=0).A1
N = len(docs)

def calcular_idf_bm25(N, df_t):
    return log((N - df_t + 0.5) / (df_t + 0.5) + 1)

idf_bm25 = {t: calcular_idf_bm25(N, df_bm25_array[i]) for i, t in enumerate(terminos_bm25)}

k1 = 1.5
b = 0.75

def score_bm25(tokens_consulta, dict_tf_doc, longitud_doc, longitud_prom, idf, k1, b):
    total = 0
    for termino in tokens_consulta:
        if termino in dict_tf_doc:
            tf = dict_tf_doc[termino]
            idf_val = idf.get(termino, 0)
            numerador = idf_val * tf * (k1 + 1)
            denominador = tf + k1 * (1 - b + b * (longitud_doc / longitud_prom))
            total += numerador / denominador
    return total

def ranking_bm25(consulta, dicts_tf, longitudes, longitud_prom, idf, k1, b, nombres):
    tokens = tokenizar(consulta)
    puntajes = []
    for i, dict_tf in enumerate(dicts_tf):
        s = score_bm25(tokens, dict_tf, longitudes[i], longitud_prom, idf, k1, b)
        puntajes.append(s)
    resultado = pd.DataFrame({'Documento': nombres, 'Score BM25': puntajes})
    return resultado.sort_values(by='Score BM25', ascending=False).reset_index(drop=True)

ranking_bm25_df = ranking_bm25(consulta, dicts_tf, longitudes, longitud_promedio, idf_bm25, k1, b, nombres)

print(f"Consulta: '{consulta}'\n")
print("Ranking por BM25:")
print(ranking_bm25_df.head(10))

print("\n--- Comparación de Rankings ---")
print("TF-IDF:")
print(ranking_tfidf.head(5))
print("\nBM25:")
print(ranking_bm25_df.head(5))

### Parte 4: Comparación visual entre TF-IDF y BM25

Este bloque utiliza `matplotlib` y `numpy` para crear un gráfico de barras que compara visualmente los scores obtenidos por TF-IDF y BM25 para la misma consulta y documento. Los pasos incluyen:

1.  **Preparación de Datos:** Extrae los scores de TF-IDF y BM25 de los DataFrames de ranking previamente calculados.
2.  **Configuración del Gráfico:** Define etiquetas, título y ancho de las barras.
3.  **Creación del Gráfico:** Genera un gráfico de barras con los dos scores.
4.  **Personalización:** Añade etiquetas a los ejes, título descriptivo y ajusta los límites del eje Y.
5.  **Etiquetas en Barras:** Muestra el valor exacto de cada score encima de su barra.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

etiquetas = ['TF-IDF Score', 'BM25 Score']
valor_tfidf = ranking_tfidf['Score Similitud'][0]
valor_bm25 = ranking_bm25_df['Score BM25'][0]

x = np.arange(len(etiquetas))
ancho = 0.35

fig, ax = plt.subplots(figsize=(8, 6))
barras = ax.bar(x, [valor_tfidf, valor_bm25], ancho, color=['skyblue', 'lightcoral'])

ax.set_ylabel('Score')
ax.set_title(f'Comparación TF-IDF vs BM25 — Consulta: "{consulta}"')
ax.set_xticks(x)
ax.set_xticklabels(etiquetas)
ax.set_ylim(bottom=0)

def etiquetar_barras(barras):
    for barra in barras:
        altura = barra.get_height()
        ax.annotate(f'{altura:.4f}',
                    xy=(barra.get_x() + barra.get_width() / 2, altura),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom')

etiquetar_barras(barras)
plt.tight_layout()
plt.show()

### Conclusiones de la Comparación TF-IDF vs BM25

Después de visualizar y comparar los scores de TF-IDF y BM25 para la consulta `"lugares turisticos ecologico aventura"`, se observa lo siguiente:

1.  **Comparación Visual de Rankings:** El gráfico de barras muestra que el **BM25 Score** es generalmente más alto que el **Score de Similitud Coseno TF-IDF** para el mismo documento. Esto indica que BM25 asigna una mayor relevancia numérica en este caso particular.

2.  **Razones de la Diferencia:** Las diferencias en los scores se deben principalmente a las distintas formas en que cada modelo calcula la relevancia:
    *   **Cálculo de IDF:** La fórmula IDF de BM25 (`log((N - DF + 0.5) / (DF + 0.5) + 1)`) produce valores no triviales incluso para términos frecuentes, mientras que el IDF de sklearn con `smooth_idf=True` tiende a valores cercanos a 1 en corpus pequeños.
    *   **Saturación de TF:** BM25 utiliza los parámetros `k1` y `b` para evitar que frecuencias de términos muy altas influyan excesivamente, normalizando además por la longitud del documento.
    *   **Mecanismo de Puntuación:** La similitud coseno de TF-IDF mide el ángulo entre vectores (valor entre 0 y 1), mientras que BM25 suma contribuciones por término de forma independiente, generando un rango de scores no acotado superiormente.